# NB14 — RQ2 Final Train and Evaluate

Compares frozen RQ2 arms **D0**, **D-Random**, and **D-Quality** with the RQ1 Direct protocol.

- **D0**: the already-frozen RQ1 Direct checkpoint. NB14 binds it and does not retrain it.
- **D-Random** and **D-Quality** CONTINUE FROM that same frozen D0 model state (learned weights, not public XLS-R+mBART reinitialization), using the same supervised G_train plus the frozen NB13 selection of that arm.
- Gold:pseudo mixing is the predeclared slot ratio **5:1**, identical for D-Random and D-Quality and independent of unique pseudo row count.
- Training uses the compute-constrained single seed **[13]**, shared by both treatment arms. NB13 selection seed 42 is a different role and is not an NB14 training seed.
- RQ2 compute-constrained amendment before final evaluation: Gold:Pseudo remains 5:1, while the training seed policy is reduced from [13, 17, 23] to the single shared seed [13] because the measured wall-clock cost of the full multi-seed protocol is infeasible on the available RTX 4090. This amendment was made before G_test evaluation and applies identically to D-Random and D-Quality.
- RQ2 final training is enabled under the frozen protocol.
  Periodic training-time evaluation uses the exact frozen 256-row RQ1 Direct validation monitor.
  Full G_validation is retained separately and is never attached to periodic training evaluation.
  **G_test** remains locked until final readiness and explicit human unlock.
- Metrics are RQ1 SacreBLEU + chrF++ with paired cluster bootstrap on `group_id`.

Current runtime flags: `RUN_REAL_TRAINING=True`, `RQ2_FINAL_FROZEN=True`, `ALLOW_G_TEST_EVALUATION=False`.
Run All can start real final D-Random / D-Quality training; G_test stays closed.


In [ ]:
import os, sys

REPO = "/workspace/citizen-assistance-system/bahnar-s2tt-thesis"

os.chdir(REPO)

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print("cwd =", os.getcwd())
print("repo_in_sys_path =", REPO in sys.path)

In [ ]:
from src.rq2_final_contract import (
    LOCKED_RQ2_EVAL_STEPS,
    LOCKED_RQ2_EVAL_STRATEGY,
    LOCKED_RQ2_SAVE_STEPS,
    LOCKED_RQ2_SAVE_STRATEGY,
)
from src.rq2_final_train import build_training_arguments

args = build_training_arguments(
    output_dir="/tmp/rq2_args_check",
    training_contract={
        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2e-5,
        "warmup_ratio": 0.05,
        "weight_decay": 0.01,
        "num_train_epochs": 3.0,
        "seed": 13,
        "dataloader_seed": 13,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "optimizer": "adamw_torch",
        "lr_scheduler_type": "linear",
        "max_grad_norm": 1.0,
        "generation_max_length": 256,
        "num_beams": 4,
        "metric_for_best_model": "eval_sacrebleu",
        "greater_is_better": True,
        "save_total_limit": 2,
        "save_steps": LOCKED_RQ2_SAVE_STEPS,
        "eval_steps": LOCKED_RQ2_EVAL_STEPS,
        "eval_strategy": LOCKED_RQ2_EVAL_STRATEGY,
        "save_strategy": LOCKED_RQ2_SAVE_STRATEGY,
    },
)

print("gradient_checkpointing =", args.gradient_checkpointing)
print("gradient_checkpointing_kwargs =", getattr(args, "gradient_checkpointing_kwargs", None))
print("dataloader_num_workers =", args.dataloader_num_workers)
print("dataloader_pin_memory =", args.dataloader_pin_memory)
print("dataloader_persistent_workers =", args.dataloader_persistent_workers)
print("dataloader_prefetch_factor =", args.dataloader_prefetch_factor)

## 1. Imports and runtime versions

In [ ]:
import json
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir() and (cand / "notebooks").is_dir():
            return cand
    raise RuntimeError("cannot locate project root")

PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rq2_final_contract import (
    ARM_D0, ARM_QUALITY, ARM_RANDOM, ARMS, FINAL_RELATIVE_DIR, Nb14Flags,
    UNLOCK_REQUIRED, assert_augmentation_fairness, assert_nb14_output_dir,
    build_arm_training_contract, compute_nb14_source_fingerprint, default_nb14_flags,
    evaluation_protocol, experiment_root_for_arm, locked_architecture, require_configured_mix_policy,
    require_configured_seed_policy, resolve_d0_best_checkpoint_dir, resolve_frozen_d0_init, verify_upstream_rq2,
)
from src.rq2_final_data import (
    compose_arm_training_rows, data_contract_payload, load_frozen_supervised_splits,
    load_pinned_nb13_arm_manifest, validation_identity, verify_equal_budget_from_nb13,
    verify_pinned_nb13_selection,
    write_arm_manifest_csv,
)
from src.rq2_final_train import (
    bind_d0_arm, build_hparams_from_yaml, ensure_arm_root, experiment_fingerprint,
    freeze_best_checkpoint, merge_training_fields, resume_or_train_arm, select_best_checkpoint,
    training_rows_to_frame, validation_history_from_checkpoints,
)
from src.rq2_final_evaluate import (
    arm_metrics, assert_paired_alignment, derive_final_readiness, generate_from_direct_checkpoint,
    load_frozen_test_frame, paired_cluster_bootstrap_arms, persist_final_artifacts,
    publish_final_generation, require_g_test_unlocked, run_g_test_inference,
)
from src.direct_runtime_paths import resolve_direct_runtime_paths

print("PROJECT_ROOT:", PROJECT_ROOT)
print("architecture:", json.dumps(locked_architecture(), indent=2, sort_keys=True)[:400])
print("source_fingerprint:", compute_nb14_source_fingerprint()["aggregate_sha256"])


## 2. Safe flags

Train-time validation uses the frozen RQ1 Direct monitor (`direct_validation_monitor.csv`, n=256).
Full G_validation (11,112) is retained for explicit full-validation evaluation and is never
attached to periodic train-time evaluation.

RQ2 final training is enabled under the frozen protocol.
D-Random and D-Quality may be trained, while G_test remains locked
until the explicit final readiness and human-unlock stage.

Current runtime flags for this final-training run:
`RUN_REAL_TRAINING=True`, `RQ2_FINAL_FROZEN=True`, `ALLOW_G_TEST_EVALUATION=False`.
Fill `DIRECT_STATE_DIR` with the RQ1 Direct durable state directory before training.

RQ2 treatment policy was predeclared before NB14 training:
Gold:Pseudo = 5:1 and compute-constrained training seed = [13].
These values are frozen for both D-Random and D-Quality and must not be
changed in response to subsequent validation or test results.


In [ ]:
RUN_REAL_TRAINING = True
EXPECTED_NB13_GENERATION_ID = "20261005T121247321787Z-3b22064c"
ALLOW_G_TEST_EVALUATION = False
RQ2_FINAL_FROZEN = True
DIRECT_STATE_DIR = "/workspace/bahnar-s2tt-thesis/bahnar_s2tt/direct_full_state/contract_7ca16f92ac0b5532"
GOLD_PSEUDO_MIX_POLICY = "configured_gold_pseudo_slot_ratio"
GOLD_SLOTS = 5
PSEUDO_SLOTS = 1
SEED_POLICY = "compute_constrained_single_seed"
SEED_POLICY_SEEDS = [13]

FLAGS = Nb14Flags(
    run_real_training=RUN_REAL_TRAINING,
    allow_g_test_evaluation=ALLOW_G_TEST_EVALUATION,
    rq2_final_frozen=RQ2_FINAL_FROZEN,
    direct_state_dir=DIRECT_STATE_DIR,
    durable_root="",
)
print(json.dumps(FLAGS.as_dict(), indent=2, sort_keys=True))
print("GOLD_PSEUDO_MIX_POLICY:", GOLD_PSEUDO_MIX_POLICY, GOLD_SLOTS, PSEUDO_SLOTS)
print("SEED_POLICY:", SEED_POLICY, SEED_POLICY_SEEDS)
if RUN_REAL_TRAINING and not RQ2_FINAL_FROZEN:
    raise RuntimeError("RUN_REAL_TRAINING requires RQ2_FINAL_FROZEN=True after review")
if ALLOW_G_TEST_EVALUATION and not RQ2_FINAL_FROZEN:
    raise RuntimeError("ALLOW_G_TEST_EVALUATION requires RQ2_FINAL_FROZEN=True")
if RUN_REAL_TRAINING:
    require_configured_mix_policy(
        {"gold_pseudo_mix_policy": GOLD_PSEUDO_MIX_POLICY, "gold_slots": GOLD_SLOTS, "pseudo_slots": PSEUDO_SLOTS},
        for_real_training=True,
    )
    require_configured_seed_policy(
        {"seed_policy": SEED_POLICY, "seed_policy_seeds": SEED_POLICY_SEEDS},
        for_real_training=True,
    )


## 3. Resolve runtime paths

In [ ]:
from src.rq1_runtime_paths import resolve_rq1_runtime_paths

RUNTIME = resolve_rq1_runtime_paths(project_root=PROJECT_ROOT)
DURABLE_RQ2_ROOT = RUNTIME.durable_root / "artifacts" / "rq2"
NB11_U_CLEAN_DIR = DURABLE_RQ2_ROOT / "u_clean"
NB12_PSEUDO_DIR = DURABLE_RQ2_ROOT / "pseudo_labels"
NB13_SELECTION_DIR = DURABLE_RQ2_ROOT / "selection"
OUT_DIR = assert_nb14_output_dir(
    DURABLE_RQ2_ROOT / "final",
    PROJECT_ROOT,
    durable_root=RUNTIME.durable_root,
)
PATHS = resolve_direct_runtime_paths(project_root=PROJECT_ROOT, durable_root=RUNTIME.durable_root)
resolved_direct = str(FLAGS.direct_state_dir or getattr(PATHS, "direct_state_dir", "") or "")
FLAGS = Nb14Flags(
    run_real_training=FLAGS.run_real_training,
    allow_g_test_evaluation=FLAGS.allow_g_test_evaluation,
    rq2_final_frozen=FLAGS.rq2_final_frozen,
    direct_state_dir=resolved_direct,
    durable_root=str(RUNTIME.durable_root),
)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DURABLE_ROOT:", RUNTIME.durable_root)
print("NB11_U_CLEAN_DIR:", NB11_U_CLEAN_DIR)
print("NB12_PSEUDO_DIR:", NB12_PSEUDO_DIR)
print("NB13_SELECTION_DIR:", NB13_SELECTION_DIR)
print("NB14 output:", OUT_DIR)
print("resolved direct_state_dir:", FLAGS.direct_state_dir)
print(json.dumps(FLAGS.as_dict(), indent=2, sort_keys=True))
if RUN_REAL_TRAINING and not FLAGS.direct_state_dir:
    raise RuntimeError("RUN_REAL_TRAINING requires a resolved D0 direct_state_dir")


## 4. Load and verify upstream contracts

In [ ]:
UPSTREAM = None
if not FLAGS.direct_state_dir:
    if RUN_REAL_TRAINING:
        raise RuntimeError("RUN_REAL_TRAINING requires a resolved D0 direct_state_dir")
    print("upstream verify: skipped (resolved D0 direct_state_dir is empty; real training cannot start)")
else:
    UPSTREAM = verify_upstream_rq2(
        PROJECT_ROOT,
        flags=FLAGS,
        artifact_root=RUNTIME.durable_root,
        expected_nb13_generation_id=EXPECTED_NB13_GENERATION_ID,
        expected_budget_hours=20.0,
        expected_random_seed=42,
    )
    if UPSTREAM["resolved_nb13_generation_id"] != EXPECTED_NB13_GENERATION_ID:
        raise RuntimeError("resolved NB13 generation does not match EXPECTED_NB13_GENERATION_ID")
    print("nb11:", UPSTREAM["nb11_generation_id"])
    print("nb12:", UPSTREAM["nb12_generation_id"])
    print("nb13:", UPSTREAM["nb13_generation_id"])
    print("nb13 CURRENT audit:", UPSTREAM["nb13_current_generation_id"])
    print("d0_policy:", UPSTREAM["d0"]["d0_policy"])
    print("d0_checkpoint_fingerprint:", UPSTREAM["d0"]["checkpoint_fingerprint_sha256"])


## 5. Build NB14 data contracts

In [ ]:
DATA = None
SUPERVISED = None
if UPSTREAM is None:
    print("data contracts: not built")
else:
    SUPERVISED = load_frozen_supervised_splits(
        PROJECT_ROOT,
        d0_identity=UPSTREAM["d0"],
        direct_state_dir=FLAGS.direct_state_dir,
        durable_root=FLAGS.durable_root or RUNTIME.durable_root,
    )
    g_train = SUPERVISED["train"]
    g_val = SUPERVISED["validation"]
    g_monitor = SUPERVISED["monitor_frame"]
    if g_monitor is None or len(g_monitor) == 0:
        raise RuntimeError("frozen RQ1 Direct validation monitor is required for train-time eval")
    NB13_SELECTION = verify_pinned_nb13_selection(PROJECT_ROOT, generation_id=UPSTREAM["nb13_generation_id"], artifact_root=RUNTIME.durable_root)
    budget = verify_equal_budget_from_nb13(NB13_SELECTION)
    val_id = validation_identity(g_val)
    DATA = {}
    for arm in (ARM_D0, ARM_RANDOM, ARM_QUALITY):
        pseudo = [] if arm == ARM_D0 else load_pinned_nb13_arm_manifest(
            PROJECT_ROOT, arm=arm, upstream=UPSTREAM, artifact_root=RUNTIME.durable_root,
        )
        composition = compose_arm_training_rows(
            arm=arm,
            supervised_rows=g_train.to_dict("records"),
            pseudo_rows=pseudo,
            selection_contract_sha256=UPSTREAM["nb13_selection_contract_sha256"],
            nb12_contract_sha256=UPSTREAM["nb12_contract_sha256"],
        )
        manifest_path = OUT_DIR / f"{arm}_train_manifest.csv"
        sha = write_arm_manifest_csv(manifest_path, composition["rows"])
        DATA[arm] = {
            "composition": composition,
            "data_contract": data_contract_payload(
                composition, manifest_sha256=sha, validation=val_id, upstream=UPSTREAM, budget=budget,
            ),
            "manifest_sha256": sha,
            "train_frame": training_rows_to_frame(composition["rows"]),
            "validation_frame": g_val,
            "monitor_frame": g_monitor,
        }
        print(
            arm,
            "rows", composition["n_rows"],
            "pseudo", composition["pseudo"]["n_rows"],
            "full_val", len(g_val),
            "eval_monitor", len(g_monitor),
        )


## 6. Build per-arm training contracts

In [ ]:
CONTRACTS = None
LAYOUTS = None
SEED_RUNS = None
D0_INIT = None
if UPSTREAM is None or DATA is None:
    print("training contracts: not built")
else:
    from src.rq2_final_contract import (
        assert_augmentation_fairness,
        compute_nb14_source_fingerprint,
        materialize_seed_bound_contracts,
    )
    FAIRNESS_PROOF = None
    hparams = build_hparams_from_yaml(PROJECT_ROOT)
    source_fp = compute_nb14_source_fingerprint()["aggregate_sha256"]
    D0_INIT = resolve_frozen_d0_init(FLAGS, project_root=PROJECT_ROOT, identity=UPSTREAM["d0"])
    mix_policy = {
        "gold_pseudo_mix_policy": GOLD_PSEUDO_MIX_POLICY,
        "gold_slots": GOLD_SLOTS,
        "pseudo_slots": PSEUDO_SLOTS,
    }
    seed_policy = {
        "seed_policy": SEED_POLICY,
        "seed_policy_seeds": list(SEED_POLICY_SEEDS),
    }
    CONTRACTS = {}
    LAYOUTS = {}
    SEED_RUNS = {}
    for arm in (ARM_D0, ARM_RANDOM, ARM_QUALITY):
        fields = merge_training_fields(
            arm=arm,
            data_contract=DATA[arm]["data_contract"],
            hparams=hparams,
            source_fingerprint_sha256=source_fp,
            rq1_direct_training_contract_hash=UPSTREAM["d0"]["direct_training_contract_hash"],
            d0_init=D0_INIT,
            mix_policy=mix_policy,
            seed_policy=seed_policy,
        )
        if arm != ARM_D0 and mix_policy["gold_pseudo_mix_policy"] == "UNSET_REQUIRE_EXPLICIT_CONFIG":
            print(arm, "training contract not frozen: GOLD_PSEUDO_MIX_POLICY is unset")
            CONTRACTS = None
            LAYOUTS = None
            SEED_RUNS = None
            break
        selected_manifest = ""
        if arm == ARM_RANDOM:
            selected_manifest = UPSTREAM["d_random_manifest_sha256"]
        elif arm == ARM_QUALITY:
            selected_manifest = UPSTREAM["d_quality_manifest_sha256"]
        bound = materialize_seed_bound_contracts(fields)
        runs = []
        for contract in bound:
            fingerprint = experiment_fingerprint(
                arm=arm,
                training_contract=contract,
                data_manifest_sha256=DATA[arm]["manifest_sha256"],
                model_revision=contract["decoder_revision"],
                selected_manifest_sha256=selected_manifest,
                nb13_generation_id=UPSTREAM["resolved_nb13_generation_id"],
            )
            root = experiment_root_for_arm(
                durable_root=RUNTIME.durable_root,
                contract_hash=contract["arm_training_contract_sha256"],
                arm=arm,
            )
            runs.append({
                "seed": contract.get("seed"),
                "contract": contract,
                "fingerprint": fingerprint,
                "layout": ensure_arm_root(root),
            })
            print(arm, "seed", contract.get("seed"), "contract", contract["arm_training_contract_sha256"][:16], "root", root.name)
        from src.rq2_final_contract import index_seed_runs
        if arm == ARM_D0:
            d0_run, = runs
            indexed = {"d0": d0_run}
        else:
            indexed = index_seed_runs(runs)
        SEED_RUNS[arm] = indexed
        shared = next(iter(indexed.values()))
        if arm == ARM_D0:
            CONTRACTS[arm] = {"contract": shared["contract"], "fingerprint": shared["fingerprint"], "seed_runs": runs, "by_seed": indexed}
            LAYOUTS[arm] = shared["layout"]
        else:
            only = shared if len(indexed) == 1 else None
            CONTRACTS[arm] = {
                "contract": None if only is None else only["contract"],
                "fingerprint": None if only is None else only["fingerprint"],
                "seed_runs": runs,
                "by_seed": indexed,
                "seed_policy": shared["contract"],
            }
            LAYOUTS[arm] = {seed: run["layout"] for seed, run in indexed.items()}
    if CONTRACTS is not None:
        random_runs = [SEED_RUNS[ARM_RANDOM][seed] for seed in SEED_RUNS[ARM_RANDOM]]
        quality_runs = [SEED_RUNS[ARM_QUALITY][seed] for seed in SEED_RUNS[ARM_QUALITY]]
        if list(SEED_RUNS[ARM_RANDOM]) != list(SEED_RUNS[ARM_QUALITY]):
            raise RuntimeError("D-Random and D-Quality must use the same declared seeds")
        proofs = []
        for left, right in zip(random_runs, quality_runs):
            if int(left["contract"]["active_seed"]) != int(right["contract"]["active_seed"]):
                raise RuntimeError("fairness pair is not the same active_seed")
            proofs.append(assert_augmentation_fairness(left["contract"], right["contract"]))
        FAIRNESS_PROOF = dict(proofs[0])
        if len(proofs) > 1:
            if len({item["fairness_proof_sha256"] for item in proofs}) != 1:
                raise RuntimeError("paired fairness proofs differ across seeds")
            FAIRNESS_PROOF["per_seed"] = proofs
        print("fairness_proof", FAIRNESS_PROOF["fairness_proof_sha256"], "n_seeds", len(proofs))
        print("d0_init_model_state", D0_INIT["model_state_sha256"])


## 7. Preflight report

In [ ]:
print("RUN_REAL_TRAINING:", RUN_REAL_TRAINING)
print("ALLOW_G_TEST_EVALUATION:", ALLOW_G_TEST_EVALUATION)
print("RQ2_FINAL_FROZEN:", RQ2_FINAL_FROZEN)
print("UNLOCK_REQUIRED:", UNLOCK_REQUIRED)

## 8. Optional REAL training gate

In [ ]:
TRUNCATION_AUDIT = None
PSEUDO_AUDIO_AUDIT = None
if not RUN_REAL_TRAINING:
    print("real training gate: closed")
else:
    if UPSTREAM is None or DATA is None or CONTRACTS is None or not SEED_RUNS:
        raise RuntimeError("refusing training: upstream, data, or training contracts were not verified")
    if not FLAGS.direct_state_dir:
        raise RuntimeError("RUN_REAL_TRAINING requires a resolved D0 direct_state_dir")
    from src.direct_model import load_direct_processors
    from src.rq2_final_train import audit_split_target_truncation, enforce_target_truncation_gate, verify_pseudo_audio_rows
    random_contracts = [run["contract"] for run in SEED_RUNS[ARM_RANDOM].values()]
    contract = random_contracts[0]
    for other in random_contracts[1:]:
        for key in ("max_target_length", "hard_max_truncation_rate", "encoder_id", "decoder_id", "gold_slots", "pseudo_slots"):
            if other.get(key) != contract.get(key):
                raise RuntimeError(f"seed runs disagree on shared training field {key}")
    _feature_extractor, tokenizer = load_direct_processors(
        encoder_id=str(contract["encoder_id"]),
        encoder_revision=str(contract["encoder_revision"]),
        decoder_id=str(contract["decoder_id"]),
        decoder_revision=str(contract["decoder_revision"]),
        target_lang=str(contract["target_lang"]),
    )
    max_target_length = int(contract["max_target_length"])

    def _texts(frame):
        if "text_vi_norm" not in frame.columns:
            raise RuntimeError("truncation audit requires canonical text_vi_norm")
        return [str(value) for value in frame["text_vi_norm"].tolist()]

    def _pseudo_rows(arm):
        return [row for row in DATA[arm]["composition"]["rows"] if row.get("kind") == "pseudo_nb13"]

    reports = [
        audit_split_target_truncation(tokenizer, _texts(SUPERVISED["train"]), split="gold_train", max_target_length=max_target_length),
        audit_split_target_truncation(tokenizer, _texts(SUPERVISED["validation"]), split="g_validation", max_target_length=max_target_length),
    ]
    PSEUDO_AUDIO_AUDIT = {}
    for arm in (ARM_RANDOM, ARM_QUALITY):
        rows = _pseudo_rows(arm)
        reports.append(audit_split_target_truncation(
            tokenizer,
            [str(row["target_text_norm"]) for row in rows],
            split=f"{arm}_pseudo",
            max_target_length=max_target_length,
        ))
        PSEUDO_AUDIO_AUDIT[arm] = verify_pseudo_audio_rows(rows, u_clean_dir=NB11_U_CLEAN_DIR)
        print(arm, "pseudo audio", PSEUDO_AUDIO_AUDIT[arm]["n_rows"])
    TRUNCATION_AUDIT = enforce_target_truncation_gate(
        reports,
        max_truncation_rate=float(contract["hard_max_truncation_rate"]),
    )
    for arm, runs in SEED_RUNS.items():
        for active_seed, run in runs.items():
            if arm != ARM_D0 and int(run["contract"]["active_seed"]) != int(active_seed):
                raise RuntimeError(f"{arm} run key {active_seed} does not match active_seed")
            (run["layout"]["root"] / "target_truncation_audit.json").write_text(
                json.dumps(TRUNCATION_AUDIT, indent=2, sort_keys=True),
                encoding="utf-8",
            )
            if arm in PSEUDO_AUDIO_AUDIT:
                (run["layout"]["root"] / "pseudo_audio_preflight.json").write_text(
                    json.dumps(PSEUDO_AUDIO_AUDIT[arm], indent=2, sort_keys=True),
                    encoding="utf-8",
                )
    print("truncation audit", TRUNCATION_AUDIT["audit_sha256"], "passed", TRUNCATION_AUDIT["passed"])


## 9. Train or resume arms

In [ ]:
TRAIN_RESULTS = {}
if CONTRACTS is None or LAYOUTS is None or DATA is None or not SEED_RUNS:
    print("train/resume: skipped (contracts not built; real training cannot start)")
else:
    from src.rq2_final_train import discover_latest_valid_checkpoint
    for arm in (ARM_D0, ARM_RANDOM, ARM_QUALITY):
        for _seed, run in SEED_RUNS[arm].items():
            contract = run["contract"]
            resume_checkpoint = None
            if arm != ARM_D0:
                selected_manifest = UPSTREAM["d_random_manifest_sha256"] if arm == ARM_RANDOM else UPSTREAM["d_quality_manifest_sha256"]
                resume_checkpoint = discover_latest_valid_checkpoint(
                    run["layout"],
                    arm=arm,
                    expected_contract_hash=contract["arm_training_contract_sha256"],
                    expected_data_hash=contract["data_manifest_sha256"],
                    expected_nb13_generation_id=UPSTREAM["resolved_nb13_generation_id"],
                    expected_manifest_sha256=contract["data_manifest_sha256"],
                    expected_selected_manifest_sha256=selected_manifest,
                    expected_d0_init_sha256=contract["d0_init_model_state_sha256"],
                    expected_active_seed=int(contract["active_seed"]),
                )
            key = f"{arm}:{run['seed']}"
            TRAIN_RESULTS[key] = resume_or_train_arm(
                arm=arm,
                flags=FLAGS,
                layout=run["layout"],
                training_contract=contract,
                fingerprint=run["fingerprint"],
                train_frame=DATA[arm]["train_frame"],
                validation_frame=DATA[arm]["validation_frame"],
                monitor_frame=DATA[arm]["monitor_frame"],
                audio_cache_roots=[PATHS.audio_cache_dir, NB11_U_CLEAN_DIR],
                resume_checkpoint=resume_checkpoint,
            )
            print(key, TRAIN_RESULTS[key].get("reason") or TRAIN_RESULTS[key].get("trained"))
            if arm != ARM_D0:
                print(
                    key,
                    "train-time eval_monitor_n=",
                    contract.get("monitor_size"),
                    "full_validation_n=",
                    len(DATA[arm]["validation_frame"]),
                    "eval_steps=",
                    contract.get("eval_steps"),
                )
    if not RUN_REAL_TRAINING:
        print("real HuggingFace Trainer.train() was not called")


## 10. Freeze best checkpoints from validation

In [ ]:
BEST = {}
if not LAYOUTS or not CONTRACTS or not SEED_RUNS:
    print("best-checkpoint freeze: skipped")
else:
    for arm in ARMS:
        if arm == ARM_D0:
            BEST[arm] = json.loads(LAYOUTS[arm]["best_checkpoint"].read_text(encoding="utf-8")) if LAYOUTS[arm]["best_checkpoint"].is_file() else {}
            continue
        frozen = {}
        for seed, run in SEED_RUNS[arm].items():
            history = validation_history_from_checkpoints(
                run["layout"],
                arm=arm,
                training_contract_sha256=run["contract"]["arm_training_contract_sha256"],
                data_contract_sha256=run["contract"]["data_manifest_sha256"],
            )
            if not history:
                print(arm, "seed", seed, "no validation history yet")
                continue
            selected = select_best_checkpoint(history)
            selected["active_seed"] = int(run["contract"]["active_seed"])
            selected["nb13_generation_id"] = UPSTREAM["resolved_nb13_generation_id"]
            selected["d0_init_model_state_sha256"] = run["contract"]["d0_init_model_state_sha256"]
            selected["selected_manifest_sha256"] = run["fingerprint"].get("selected_manifest_sha256")
            frozen[int(seed)] = freeze_best_checkpoint(
                run["layout"],
                selected,
                arm=arm,
                contract_hash=run["contract"]["arm_training_contract_sha256"],
                data_hash=run["contract"]["data_manifest_sha256"],
            )
            print(arm, "seed", seed, frozen[int(seed)]["checkpoint_name"], frozen[int(seed)]["checkpoint_fingerprint"])
        if frozen:
            BEST[arm] = frozen


## 11. Final evaluation readiness report

In [ ]:
PROTOCOL = evaluation_protocol(UPSTREAM["rq1_eval"]) if UPSTREAM else None
READINESS = derive_final_readiness(
    flags=FLAGS,
    upstream=UPSTREAM,
    layouts=LAYOUTS,
    contracts=CONTRACTS,
    best=BEST if "BEST" in dir() else None,
    protocol=PROTOCOL,
)
print(json.dumps({"flat": READINESS["flat"], "all_ok": READINESS["all_ok"]}, indent=2, sort_keys=True))

## 12. Explicit G_test unlock gate

In [ ]:
if ALLOW_G_TEST_EVALUATION:
    require_g_test_unlocked(
        FLAGS,
        READINESS,
        layouts=LAYOUTS,
        contracts=CONTRACTS,
        best=BEST,
        d0_identity=UPSTREAM["d0"] if UPSTREAM else None,
        upstream=UPSTREAM,
    )
    print("G_test unlock: accepted")
else:
    print("G_test unlock: blocked (ALLOW_G_TEST_EVALUATION=False)")


## 13. G_test inference

In [ ]:
PER_SEED_EVAL = None
TEST_FRAME = None
if not ALLOW_G_TEST_EVALUATION:
    print("G_test inference: skipped")
else:
    from src.rq2_final_evaluate import plan_paired_seed_evaluations
    seed_contract = next(iter(SEED_RUNS[ARM_RANDOM].values()))["contract"]
    CONTRACTS[ARM_RANDOM]["seed_policy"] = seed_contract
    pairs = plan_paired_seed_evaluations(BEST, seed_policy=seed_contract)
    eval_layouts = dict(LAYOUTS)
    if LAYOUTS.get(ARM_RANDOM) and "checkpoints" not in (LAYOUTS.get(ARM_RANDOM) or {}):
        eval_layouts = {
            ARM_D0: SEED_RUNS[ARM_D0]["d0"]["layout"],
            ARM_RANDOM: LAYOUTS[ARM_RANDOM],
            ARM_QUALITY: LAYOUTS[ARM_QUALITY],
        }
    require_g_test_unlocked(
        FLAGS, READINESS, layouts=eval_layouts, contracts=CONTRACTS, best=BEST,
        d0_identity=UPSTREAM["d0"], upstream=UPSTREAM,
    )
    TEST_FRAME = load_frozen_test_frame(
        project_root=PROJECT_ROOT, flags=FLAGS, readiness=READINESS,
        layouts=eval_layouts, contracts=CONTRACTS, best=BEST,
        d0_identity=UPSTREAM["d0"], upstream=UPSTREAM,
    )

    def _record(arm, seed):
        payload = BEST[arm]
        if "checkpoint_fingerprint" in payload:
            return payload
        return payload[seed]

    def _one_frame(arm, payload, contract, layout):
        fingerprint = str(payload.get("checkpoint_fingerprint") or "")
        ckpt_name = str(payload.get("checkpoint_name") or "")
        if arm == ARM_D0:
            ckpt_dir = resolve_d0_best_checkpoint_dir(FLAGS.direct_state_dir, UPSTREAM["d0"])
        else:
            ckpt_dir = layout["checkpoints"] / ckpt_name
        generate_fn = generate_from_direct_checkpoint(
            checkpoint_dir=ckpt_dir,
            training_contract=contract,
            audio_cache_roots=[PATHS.audio_cache_dir, NB11_U_CLEAN_DIR],
        )
        return run_g_test_inference(
            arm=arm,
            test_frame=TEST_FRAME,
            checkpoint_fingerprint=fingerprint,
            generate_fn=generate_fn,
        )

    d0_run = SEED_RUNS[ARM_D0]["d0"]
    d0_frame = _one_frame(ARM_D0, BEST[ARM_D0], d0_run["contract"], d0_run["layout"])
    PER_SEED_EVAL = {}
    for pair in pairs:
        seed = int(pair["active_seed"])
        seed_frames = {ARM_D0: d0_frame}
        for arm in (ARM_RANDOM, ARM_QUALITY):
            payload = _record(arm, seed)
            run = SEED_RUNS[arm][seed]
            if int(payload.get("active_seed")) != seed:
                raise RuntimeError(f"{arm} evaluation seed {payload.get('active_seed')} is not {seed}")
            seed_frames[arm] = _one_frame(arm, payload, run["contract"], run["layout"])
        assert_paired_alignment(seed_frames)
        metrics = {arm: arm_metrics(seed_frames[arm]) for arm in (ARM_RANDOM, ARM_QUALITY)}
        bootstrap = paired_cluster_bootstrap_arms(
            seed_frames,
            n_samples=int(PROTOCOL["bootstrap_samples"]),
            seed=int(PROTOCOL["seed"]),
            confidence=float(PROTOCOL["confidence"]),
        )
        PER_SEED_EVAL[seed] = {
            "active_seed": seed,
            "pairing": pair["pairing"],
            "random": {
                "arm": ARM_RANDOM,
                "active_seed": seed,
                "checkpoint_name": str(_record(ARM_RANDOM, seed).get("checkpoint_name") or ""),
                "checkpoint_fingerprint": str(_record(ARM_RANDOM, seed)["checkpoint_fingerprint"]),
                "metrics": metrics[ARM_RANDOM],
            },
            "quality": {
                "arm": ARM_QUALITY,
                "active_seed": seed,
                "checkpoint_name": str(_record(ARM_QUALITY, seed).get("checkpoint_name") or ""),
                "checkpoint_fingerprint": str(_record(ARM_QUALITY, seed)["checkpoint_fingerprint"]),
                "metrics": metrics[ARM_QUALITY],
            },
            "paired": {
                "frames": seed_frames,
                "bootstrap": bootstrap,
                "delta_sacrebleu": float(metrics[ARM_QUALITY]["sacrebleu"]) - float(metrics[ARM_RANDOM]["sacrebleu"]),
                "delta_chrfpp": float(metrics[ARM_QUALITY]["chrfpp"]) - float(metrics[ARM_RANDOM]["chrfpp"]),
            },
            ARM_RANDOM: {
                "sacrebleu": metrics[ARM_RANDOM]["sacrebleu"],
                "chrfpp": metrics[ARM_RANDOM]["chrfpp"],
                "arm": ARM_RANDOM,
                "active_seed": seed,
                "checkpoint_name": str(_record(ARM_RANDOM, seed).get("checkpoint_name") or ""),
                "checkpoint_fingerprint": str(_record(ARM_RANDOM, seed)["checkpoint_fingerprint"]),
            },
            ARM_QUALITY: {
                "sacrebleu": metrics[ARM_QUALITY]["sacrebleu"],
                "chrfpp": metrics[ARM_QUALITY]["chrfpp"],
                "arm": ARM_QUALITY,
                "active_seed": seed,
                "checkpoint_name": str(_record(ARM_QUALITY, seed).get("checkpoint_name") or ""),
                "checkpoint_fingerprint": str(_record(ARM_QUALITY, seed)["checkpoint_fingerprint"]),
            },
        }
        print("paired seed", seed, "delta_sacrebleu", PER_SEED_EVAL[seed]["paired"]["delta_sacrebleu"])
    print("per-seed evaluations", sorted(PER_SEED_EVAL), "no best seed was selected from G_test")


## 14. Metrics

In [ ]:
METRICS = None
SEED_SUMMARY = None
if not PER_SEED_EVAL:
    print("metrics: skipped")
else:
    from src.rq2_final_contract import aggregate_paired_seed_metrics
    for seed, payload in PER_SEED_EVAL.items():
        assert_paired_alignment(payload["paired"]["frames"])
        if int(payload["active_seed"]) != int(seed):
            raise RuntimeError(f"PER_SEED_EVAL key {seed} does not match active_seed")
    seed_contract = next(iter(SEED_RUNS[ARM_RANDOM].values()))["contract"]
    SEED_SUMMARY = aggregate_paired_seed_metrics(list(PER_SEED_EVAL.values()), seed_policy=seed_contract)
    METRICS = SEED_SUMMARY
    print("n_valid_runs", SEED_SUMMARY["n_valid_runs"], "std_convention", SEED_SUMMARY["std_convention"])


## 15. Paired cluster bootstrap

In [ ]:
BOOTSTRAP = None
if not PER_SEED_EVAL or PROTOCOL is None:
    print("bootstrap: skipped")
else:
    BOOTSTRAP = {int(seed): payload["paired"]["bootstrap"] for seed, payload in PER_SEED_EVAL.items()}
    print("per-seed bootstrap", sorted(BOOTSTRAP))


## 16. Final contract and artifact verification

In [ ]:
ARTIFACTS = None
if not PER_SEED_EVAL or BOOTSTRAP is None or UPSTREAM is None or SEED_SUMMARY is None:
    print("final contract: skipped (G_test not evaluated)")
else:
    from src.rq1_contract import ordered_uid_hash, reference_content_hash, sha256_file
    from src.rq2_final_contract import STATUS_TRAINING_COMPLETE, build_final_contract
    test_path = PROJECT_ROOT / "data" / "manifests" / "rq1_test.csv"
    per_seed_frames = {int(seed): payload["paired"]["frames"] for seed, payload in PER_SEED_EVAL.items()}
    legacy_frames = per_seed_frames[next(iter(per_seed_frames))]

    def _training_bundle(arm):
        layouts = LAYOUTS[arm]
        runs = []
        for seed, layout in layouts.items():
            payload = json.loads(layout["training_complete"].read_text(encoding="utf-8"))
            payload["active_seed"] = int(payload.get("active_seed") or seed)
            runs.append(payload)
        if len(runs) == 1:
            only_run, = runs
            one = dict(only_run)
            one["seed_runs"] = runs
            one["status"] = STATUS_TRAINING_COMPLETE
            return one
        return {
            "status": STATUS_TRAINING_COMPLETE,
            "arm": arm,
            "g_test_used": False,
            "seed_runs": runs,
        }

    fields = {
        "nb11_generation_id": UPSTREAM["nb11_generation_id"],
        "nb11_input_contract_sha256": UPSTREAM["nb11_input_contract_sha256"],
        "nb12_generation_id": UPSTREAM["nb12_generation_id"],
        "nb12_contract_sha256": UPSTREAM["nb12_contract_sha256"],
        "nb13_selection_contract_sha256": UPSTREAM["nb13_selection_contract_sha256"],
        "nb13_generation_id": UPSTREAM["nb13_generation_id"],
        "d_random_manifest_sha256": UPSTREAM["d_random_manifest_sha256"],
        "d_quality_manifest_sha256": UPSTREAM["d_quality_manifest_sha256"],
        "d0_direct_training_contract_hash": UPSTREAM["d0"]["direct_training_contract_hash"],
        "d0_checkpoint_fingerprint": BEST[ARM_D0]["checkpoint_fingerprint"],
        "d0_init_model_state_sha256": D0_INIT["model_state_sha256"],
        "seed_policy_record": next(iter(SEED_RUNS[ARM_RANDOM].values()))["contract"]["seed_policy_record"],
        "gold_pseudo_mix": {
            "policy": GOLD_PSEUDO_MIX_POLICY,
            "gold_slots": GOLD_SLOTS,
            "pseudo_slots": PSEUDO_SLOTS,
        },
        "expected_nb13_generation_id": EXPECTED_NB13_GENERATION_ID,
        "resolved_nb13_generation_id": UPSTREAM["resolved_nb13_generation_id"],
        "g_test_manifest_sha256": sha256_file(test_path),
        "g_test_ordered_uid_hash": ordered_uid_hash(TEST_FRAME),
        "g_test_reference_hash": reference_content_hash(TEST_FRAME),
        "source_fingerprint_sha256": compute_nb14_source_fingerprint()["aggregate_sha256"],
        "bootstrap_config": PROTOCOL,
        "fairness_proof": FAIRNESS_PROOF,
        "d_random_training_complete": _training_bundle(ARM_RANDOM),
        "d_quality_training_complete": _training_bundle(ARM_QUALITY),
    }
    if len(PER_SEED_EVAL) == 1:
        seed = next(iter(PER_SEED_EVAL))
        fields["d_random_best_checkpoint_fingerprint"] = BEST[ARM_RANDOM][seed]["checkpoint_fingerprint"]
        fields["d_quality_best_checkpoint_fingerprint"] = BEST[ARM_QUALITY][seed]["checkpoint_fingerprint"]
        fields["d_random_training_contract_sha256"] = SEED_RUNS[ARM_RANDOM][seed]["contract"]["arm_training_contract_sha256"]
        fields["d_quality_training_contract_sha256"] = SEED_RUNS[ARM_QUALITY][seed]["contract"]["arm_training_contract_sha256"]
        fields["d_random_best_checkpoint"] = BEST[ARM_RANDOM][seed]["checkpoint_name"]
        fields["d_quality_best_checkpoint"] = BEST[ARM_QUALITY][seed]["checkpoint_name"]
    ARTIFACTS = persist_final_artifacts(
        OUT_DIR / "_stage_artifacts",
        frames=legacy_frames,
        per_seed_frames=per_seed_frames,
        contract_fields=fields,
        best=BEST,
        n_samples=int(PROTOCOL["bootstrap_samples"]),
        seed=int(PROTOCOL["seed"]),
        confidence=float(PROTOCOL["confidence"]),
    )
    print("staged artifacts", sorted(ARTIFACTS))


## 17. Publish CURRENT only on success

In [ ]:
if ARTIFACTS is None or not ALLOW_G_TEST_EVALUATION or not RQ2_FINAL_FROZEN:
    print("publication: skipped unless every success gate is true")
else:
    published = publish_final_generation(PROJECT_ROOT, artifacts=ARTIFACTS, durable_root=str(RUNTIME.durable_root), artifact_root=RUNTIME.durable_root)
    print(json.dumps({"status": published["status"], "wrote_current": published["wrote_current"]}, indent=2))
    if published["status"] != "SUCCESS_RQ2_FINAL":
        raise RuntimeError(published.get("error") or "NB14 publication failed independent verification")
print("RUN_REAL_TRAINING:", RUN_REAL_TRAINING)
print("ALLOW_G_TEST_EVALUATION:", ALLOW_G_TEST_EVALUATION)
print("RQ2_FINAL_FROZEN:", RQ2_FINAL_FROZEN)
